<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/01_nivelacion_ml/12_arboles_bosques_bagging.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Árboles, bosques y bagging

**Pregunta guía:** ¿Cómo reduce varianza un conjunto de árboles?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## Matemática esencial

Un árbol elige cortes que reducen impureza. Para clases con proporciones
$p_k$, Gini es $G=1-\sum_k p_k^2$. Un árbol profundo tiene poco sesgo y
alta varianza. Bagging entrena estimadores sobre muestras bootstrap y
promedia sus predicciones. Si los errores tienen correlación $\rho$, la
varianza del promedio de $B$ modelos se aproxima por
$\sigma^2[\rho+(1-\rho)/B]$. Random Forest también submuestrea variables
para reducir $\rho$.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

SEMILLA = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)


In [ ]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

X, y = make_classification(
    n_samples=1_200,
    n_features=12,
    n_informative=6,
    n_redundant=3,
    class_sep=1.0,
    flip_y=0.04,
    random_state=SEMILLA,
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA
)
modelos = {
    "árbol": (
        DecisionTreeClassifier(random_state=SEMILLA),
        {"max_depth": [2, 4, 8, None], "min_samples_leaf": [1, 5, 15]},
    ),
    "bagging": (
        BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=SEMILLA),
            random_state=SEMILLA,
            n_jobs=-1,
        ),
        {"n_estimators": [30, 100], "max_samples": [0.6, 1.0]},
    ),
    "random_forest": (
        RandomForestClassifier(random_state=SEMILLA, n_jobs=-1),
        {
            "n_estimators": [100, 250],
            "max_depth": [4, 10, None],
            "max_features": ["sqrt", 0.7],
            "min_samples_leaf": [1, 5],
        },
    ),
}

filas, búsquedas = [], {}
for nombre, (modelo, grilla) in modelos.items():
    búsqueda = GridSearchCV(
        modelo, grilla, scoring="f1", cv=cv, n_jobs=-1, return_train_score=True
    ).fit(X_dev, y_dev)
    pred = búsqueda.predict(X_test)
    búsquedas[nombre] = búsqueda
    filas.append(
        {
            "modelo": nombre,
            "F1_CV": búsqueda.best_score_,
            "F1_test": f1_score(y_test, pred),
            "accuracy_test": accuracy_score(y_test, pred),
            "mejor": búsqueda.best_params_,
        }
    )
display(pd.DataFrame(filas).set_index("modelo"))


In [ ]:
árbol_pequeño = DecisionTreeClassifier(max_depth=3, random_state=SEMILLA).fit(
    X_dev, y_dev
)
plt.figure(figsize=(16, 6))
plot_tree(árbol_pequeño, max_depth=2, filled=True, fontsize=8)
plt.title("Primeras decisiones de un árbol limitado")
plt.show()

bosque = búsquedas["random_forest"].best_estimator_
importancia = pd.Series(bosque.feature_importances_).sort_values(ascending=False)
importancia.head(10).sort_values().plot.barh(title="Importancia por impureza")
plt.xlabel("reducción media de impureza")
plt.show()


**Advertencia:** la importancia por impureza puede favorecer variables
continuas o con muchas categorías; se contrastará con permutación en el
notebook 15.

**Ejercicios:** observe la brecha train–CV al variar profundidad; mida
cuánto cambia el bosque con cinco semillas; explique por qué aumentar
árboles reduce varianza pero no corrige un sesgo sistemático.
